# Completed local extensions for the final revision

This notebook contains the three fitted-model add-on experiments used in the final revision:

1. a ten-seed size-by-balance factorial on CIC-UNSW-NB15 V1;
2. data-rich Random Forest and XGBoost references for the grouped-CIC Axis-2 rows; and
3. the ten-seed TabPFN 2.5 ManyClass sensitivity analysis.

The completed second hosted-model Axis-2 panel is recorded separately in `revision4_experiments_completed.ipynb` and under `results/round4/raw/axis2_second_variant/`. The persisted CSV/JSON files are the canonical record used by the article. Re-execution of the fitted-model sections requires the benchmark sources; the ManyClass section additionally requires a CUDA GPU and a Hugging Face token.


In [ ]:
# ============================ SWITCHES ============================
RUN_2X2_UNSW       = True
RUN_AXIS2_DATARICH = True
RUN_MANYCLASS_10   = False   # expensive; set True only for a deliberate GPU rerun

SEEDS   = list(range(10))
OUT_DIR = "revision4_outputs"
REPO_DIR = "."               # run from the repository root
MANUSCRIPT_DIR = None

NEEDS_REPO   = RUN_2X2_UNSW or RUN_MANYCLASS_10
NEEDS_MODELS = RUN_MANYCLASS_10
NEEDS_KAGGLE = RUN_AXIS2_DATARICH
print(f"repo={NEEDS_REPO} model stack={NEEDS_MODELS} kaggle={NEEDS_KAGGLE}")


## 0. Dependencies and helpers

Identical to the released sensitivity notebook (pinned model stack for TabPFN), plus a Student-$t$ interval and a Holm helper for the new analyses.

In [ ]:
import subprocess
import sys


def pip_install(specs):
    if specs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *specs], check=True)

pip_install(["pandas", "numpy", "scipy", "scikit-learn", "xgboost", "pyarrow", "kagglehub", "huggingface-hub", "requests"])
if NEEDS_MODELS:
    pip_install([
        "transformers==4.57.1", "tokenizers==0.22.1", "safetensors==0.7.0",
        "huggingface-hub==0.36.0", "accelerate==1.11.0", "peft==0.18.0",
        "tabpfn==6.0.6", "tabpfn-extensions==0.2.1",
        "tabpfn-common-utils==0.2.10",
    ])
print("dependencies ready")


In [ ]:
import gc, importlib, itertools, json, os, re, shutil, sys, time, types
from pathlib import Path
import numpy as np, pandas as pd

OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)

# --- secrets, from Colab or the environment ------------------------------
try:
    from google.colab import userdata
    for k in ("HF_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception: pass
except Exception: pass

def sf(d):
    """Exact two-sided paired sign-flip p. Floor at n pairs is 2**-(n-1)."""
    d = np.asarray(d, float); o = abs(d.mean())
    return float(np.mean([abs(np.mean(d*np.array(x))) >= o-1e-15
                          for x in itertools.product((-1,1), repeat=len(d))]))
def ci(d, n=10000):
    d = np.asarray(d, float); b = np.random.default_rng(0).choice(d,(n,len(d))).mean(1)
    return tuple(np.percentile(b,[2.5,97.5]))
def save(df, name):
    df.to_csv(OUT/name, index=False); print(f"[saved] {name} ({len(df)} rows)"); return df
print("output:", OUT.resolve())

from scipy import stats as _st
def tci(x, conf=0.95):
    """Student-t interval for a mean (used when fewer than ten seeds are available)."""
    x = np.asarray(x, float); n = len(x)
    if n < 2: return (np.nan, np.nan)
    h = _st.t.ppf(0.5+conf/2, n-1) * x.std(ddof=1)/np.sqrt(n)
    return (float(x.mean()-h), float(x.mean()+h))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty_like(p); run = 0.0
    for i, idx in enumerate(o):
        run = max(run, (len(p)-i)*p[idx]); out[idx] = min(1.0, run)
    return out
print("tci, holm ready")


## 1. Repository and dataset loaders *(experiments 5, 4b and 7)*

The released repository provides the persisted results the notebook verifies and resumes from (`results/axis2`, `results/sensitivity`, `results/derived`). The public release does not ship the dataset-loader package of the development tree, so the loaders are **vendored** in the next cell as a faithful transcription of `src/data`: same cleaning steps, same processed-file names, same cache paths. Each cleaned table is checked against the row and class counts stated in the manuscript (CIC-IDS2017: 2,522,362 rows, 15 native / 9 grouped classes; CIC-UNSW-NB15: 3,540,241 rows, 10 classes); a mismatch aborts (`STRICT_COUNTS`). The two shims for the pinned model stack are unchanged from the released sensitivity notebook.

In [ ]:
if NEEDS_REPO:
    REPO = Path(REPO_DIR).resolve()
    assert (REPO / "results").is_dir(), (
        f"Run this notebook from the repository root or set REPO_DIR; got {REPO}"
    )
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    print(f"repository: {REPO}")

    # shim 1: tabpfn 8.x renamed the preprocessor factory
    try:
        import tabpfn.inference_config as _ic
        for _old, _new in [("default_classifier_preprocessor_configs",
                            ["v2_5_classifier_preprocessor_configs","v2_classifier_preprocessor_configs"]),
                           ("default_regressor_preprocessor_configs",
                            ["v2_5_regressor_preprocessor_configs","v2_regressor_preprocessor_configs"])]:
            if not hasattr(_ic,_old):
                for _n in _new:
                    if hasattr(_ic,_n): setattr(_ic,_old,getattr(_ic,_n)); break
    except ImportError: pass

    # shim 2: huggingface_hub >= 1.0 removed HfFolder
    try:
        import huggingface_hub as _hh
        if not hasattr(_hh,"HfFolder"):
            class HfFolder:
                path_token = str(Path.home()/".cache/huggingface/token")
                @classmethod
                def get_token(cls):
                    try: return _hh.get_token()
                    except Exception: return None
                @classmethod
                def save_token(cls,t):
                    p=Path(cls.path_token); p.parent.mkdir(parents=True,exist_ok=True); p.write_text(t or "")
                @classmethod
                def delete_token(cls): Path(cls.path_token).unlink(missing_ok=True)
            _hh.HfFolder = HfFolder
    except ImportError: pass

    # ---- vendored dataset loaders (transcription of src/data) --------------------
    import requests as _rq

    DATA_DIR = "data"
    STRICT_COUNTS = True     # verify cleaned-table sizes against the manuscript
    UNSW_LOCAL_CSV = None    # set to a local CICFlowMeter_out.csv to skip the download
    CICUNSW_URL = "http://cicresearch.ca/CICDataset/CIC-UNSW/Dataset/CICFlowMeter_out.csv"

    CIC_IDS2017_CLASSES_MAPPING = {
        'BENIGN': 'BENIGN', 'DDoS': 'DDoS', 'DoS Hulk': 'DoS',
        'DoS GoldenEye': 'DoS', 'DoS slowloris': 'DoS', 'DoS Slowhttptest': 'DoS',
        'PortScan': 'Port Scan', 'FTP-Patator': 'Brute Force',
        'SSH-Patator': 'Brute Force', 'Bot': 'Bot',
        'Web Attack  Brute Force': 'Web Attack',
        'Web Attack  Sql Injection': 'Web Attack',
        'Web Attack  XSS': 'Web Attack',
        'Infiltration': 'Infiltration', 'Heartbleed': 'Heartbleed'}

    # expected cleaned-table sizes (Section 3.2 of the manuscript)
    _EXPECTED = {("cic", False): (2_522_362, 15), ("cic", True): (2_522_362, 9),
                 ("unsw", None): (3_540_241, 10)}

    def _features_correction(d):
        d.rename(columns={c: str(c).strip().replace(" ", "_") for c in d.columns},
                 inplace=True)
    def _ascii_correction(d, col):
        d[col] = d[col].apply(lambda x: re.sub(r'[^\x00-\x7F]+', '', str(x)))
    def _downcast(d):
        for col in d.columns:
            t = d[col].dtype
            if t != object:
                lo, hi = d[col].min(), d[col].max()
                if 'float' in str(t) and lo > np.finfo(np.float32).min and hi < np.finfo(np.float32).max:
                    try: d[col] = d[col].astype(np.float32)
                    except ValueError: pass
                elif 'int' in str(t) and lo > np.iinfo(np.int32).min and hi < np.iinfo(np.int32).max:
                    d[col] = d[col].astype(np.int32)
    def _drop_single_unique(d):
        nu = d.nunique()
        return d[nu[nu > 1].index]
    def _check_counts(tag, d):
        exp = _EXPECTED.get(tag)
        if exp is None: return
        rows, classes = exp
        ok = (len(d) == rows) and (d["Label"].nunique() == classes)
        msg = (f"{tag}: cleaned table has {len(d):,} rows / {d['Label'].nunique()} classes, "
               f"manuscript states {rows:,} / {classes}")
        if ok: print("  count check OK -", msg)
        elif STRICT_COUNTS:
            raise AssertionError(msg + "  (raw sources changed? set STRICT_COUNTS=False only "
                                       "if you understand why, since every number depends on this table)")
        else: print("  WARNING -", msg)

    class CICIDS2017:
        """Vendored transcription of src.data CICIDS2017 (pca / classes_mapping config)."""
        dataset_folder = "MachineLearningCVE"
        def __init__(self, pca=True, classes_mapping=True, **kw):
            self.default_config = {"classes_mapping": classes_mapping, "pca": pca}
            self.raw_data_dir = os.path.abspath(os.path.join(DATA_DIR, "CIC-IDS2017", self.dataset_folder))
            self.processed_data_dir = os.path.abspath(os.path.join(DATA_DIR, "CIC-IDS2017", "processed"))
            name = "cic_ids2017_" + "_".join(f"{k}({v})" for k, v in sorted(self.default_config.items()))
            self.file_path = os.path.join(self.processed_data_dir, f"{name}.csv")
            self.data = None
        def _collect(self):
            if os.path.isfile(self.file_path):
                self.data = pd.read_csv(self.file_path); return
            raw = Path(self.raw_data_dir)
            files = [raw/f for f in CIC_FILES if (raw/f).is_file()]
            files += sorted(p for p in raw.glob("*.csv") if p not in files)
            assert len(files) >= 8, (f"expected the 8 raw CIC-IDS2017 CSVs in {raw}; "
                                     f"found {len(files)} (run _prefetch_cic / set LOCAL_CIC)")
            # deterministic capture-day order (the order _prefetch_cic writes them)
            self.data = pd.concat((pd.read_csv(f, encoding="latin-1") for f in files),
                                  ignore_index=False)
        def _preprocess(self):
            _features_correction(self.data)
            _ascii_correction(self.data, "Label")
            self.data.drop_duplicates(inplace=True)
            self.data.replace([np.inf, -np.inf], np.nan, inplace=True)
            for c in ("Flow_Bytes/s", "Flow_Packets/s"):
                if c in self.data.columns:
                    self.data[c] = self.data[c].fillna(self.data[c].median())
            self.data.dropna(inplace=True)
            _downcast(self.data)
            self.data = _drop_single_unique(self.data)
            if self.default_config["pca"]:
                raise NotImplementedError("the vendored loader is used with pca=False only; "
                                          "PCA variants are built by transform(use_pca=True)")
            if self.default_config["classes_mapping"]:
                self.data["Label"] = self.data["Label"].map(CIC_IDS2017_CLASSES_MAPPING)
        def load(self):
            fresh = not os.path.isfile(self.file_path)
            self._collect()
            if fresh:
                self._preprocess()
                Path(self.processed_data_dir).mkdir(parents=True, exist_ok=True)
                self.data.to_csv(self.file_path, index=False)
            _check_counts(("cic", self.default_config["classes_mapping"]), self.data)

    class CIC_UNSW:
        """Vendored transcription of src.data CIC_UNSW (CICFlowMeter re-derivation)."""
        dataset_folder = "CIC-UNSW"
        def __init__(self, pca=True, **kw):
            self.config = {"pca": pca}
            self.raw_data_dir = os.path.abspath(os.path.join(DATA_DIR, self.dataset_folder, "raw"))
            self.processed_data_dir = os.path.abspath(os.path.join(DATA_DIR, self.dataset_folder, "processed"))
            name = "cic_unsw_" + "_".join(f"{k}({v})" for k, v in sorted(self.config.items()))
            self.file_path = os.path.join(self.processed_data_dir, f"{name}.csv")
            self.data = None
        def _download(self):
            os.makedirs(self.raw_data_dir, exist_ok=True)
            csv_path = os.path.join(self.raw_data_dir, os.path.basename(CICUNSW_URL))
            if os.path.isfile(csv_path): return
            if UNSW_LOCAL_CSV:
                shutil.copy2(UNSW_LOCAL_CSV, csv_path); return
            print(f"  downloading {CICUNSW_URL} (~1.9 GB, one time)...")
            r = _rq.get(CICUNSW_URL, stream=True, timeout=120)
            assert r.status_code == 200, f"download failed: HTTP {r.status_code}"
            with open(csv_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
            head = open(csv_path, "rb").read(200)
            assert os.path.getsize(csv_path) > 10_000_000 and not head.lstrip().startswith(b"<"), (
                "the CIC host returned an error page instead of the CSV; retry, or place the "
                "file manually and set UNSW_LOCAL_CSV")
        def _clean(self):
            files = [f for f in os.listdir(self.raw_data_dir) if f.endswith(".csv")]
            assert len(files) == 1, f"expected one raw CSV in {self.raw_data_dir}, got {len(files)}"
            df = pd.read_csv(os.path.join(self.raw_data_dir, files[0]))
            labels = df["Label"]
            df = df.select_dtypes(include=["number"])
            df["Label"] = labels
            return df
        def _preprocess(self):
            _features_correction(self.data)
            _downcast(self.data)
            self.data = _drop_single_unique(self.data)
            if self.config["pca"]:
                raise NotImplementedError("the vendored loader is used with pca=False only")
        def load(self):
            if os.path.isfile(self.file_path):
                self.data = pd.read_csv(self.file_path)
            else:
                self._download()
                self.data = self._clean()
                self._preprocess()
                Path(self.processed_data_dir).mkdir(parents=True, exist_ok=True)
                self.data.to_csv(self.file_path, index=False)
            _check_counts(("unsw", None), self.data)

    print("vendored loaders ready (CICIDS2017, CIC_UNSW)")
    print(f"pandas {pd.__version__}" + ("  (pin pandas<3 for numbers that go in the paper)"
                                        if int(pd.__version__.split('.')[0])>=3 else ""))
else:
    print("repository not needed for the enabled experiments")


In [ ]:
# ============================================================
# PATCH — robust CIC-UNSW-NB15 raw-file acquisition
# Run after the vendored-loader cell and before Experiment 5.
# ============================================================
#
# The original CIC URL sometimes returns an HTML error page with HTTP 200.
# The original loader then leaves that response at the .csv path and treats it
# as cached on the next run. This replacement:
#   1. rejects/removes that stale response;
#   2. downloads the same CICFlowMeter_out.csv from a pinned HF dataset commit;
#   3. verifies the exact byte count and SHA-256; and
#   4. makes _clean read that exact file, rather than any arbitrary CSV nearby.

if NEEDS_REPO:
    from pathlib import Path
    import hashlib
    import shutil

    UNSW_HF_REPO = "bencorn/CIC-UNSW-NB15"
    UNSW_HF_REVISION = "8b5e371bf1cfb32683a958f62acacc6be19b5c58"
    UNSW_HF_FILE = "CICFlowMeter_out.csv"
    UNSW_EXPECTED_BYTES = 1_914_305_081
    UNSW_EXPECTED_SHA256 = (
        "025a94239f29521e154be3a65d65268404d4c58600b66b94a4235eae87dbd8b5"
    )

    def _sha256_stream(path, chunk_size=8 << 20):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def _looks_like_unsw_csv(path, *, require_exact_size=True):
        path = Path(path)
        if not path.is_file():
            return False, "missing"
        size = path.stat().st_size
        if require_exact_size and size != UNSW_EXPECTED_BYTES:
            return False, f"wrong size ({size:,} bytes)"
        if size < 10_000_000:
            return False, f"too small ({size:,} bytes)"
        with path.open("rb") as handle:
            head = handle.read(65_536).lstrip()
        if head.startswith((b"<", b"<!DOCTYPE", b"<?xml")):
            return False, "HTML/XML response"
        if b"Label" not in head:
            return False, "CSV header does not contain Label"
        return True, "ok"

    def _patched_unsw_download(self):
        from huggingface_hub import hf_hub_download

        raw_dir = Path(self.raw_data_dir)
        raw_dir.mkdir(parents=True, exist_ok=True)
        target = raw_dir / UNSW_HF_FILE
        marker = raw_dir / f"{UNSW_HF_FILE}.sha256-ok"

        # Honour a manually supplied file, but validate it before copying.
        if UNSW_LOCAL_CSV:
            source = Path(UNSW_LOCAL_CSV).expanduser().resolve()
            ok, why = _looks_like_unsw_csv(source)
            if not ok:
                raise AssertionError(
                    f"UNSW_LOCAL_CSV is not the expected CICFlowMeter_out.csv: {why}; "
                    f"path={source}"
                )
            if source != target.resolve():
                tmp = target.with_suffix(target.suffix + ".copying")
                tmp.unlink(missing_ok=True)
                shutil.copy2(source, tmp)
                tmp.replace(target)
            marker.unlink(missing_ok=True)

        ok, why = _looks_like_unsw_csv(target)
        if not ok:
            # A marker belongs only to the exact file that was previously hashed.
            marker.unlink(missing_ok=True)
            if target.exists():
                bad = target.with_name(target.name + ".bad-response")
                bad.unlink(missing_ok=True)
                target.replace(bad)
                print(f"  removed invalid cached response ({why}): {bad}")

            print(
                "  downloading pinned mirror "
                f"{UNSW_HF_REPO}/{UNSW_HF_FILE} (~1.9 GB, one time)..."
            )

            last_error = None
            for force in (False, True):
                try:
                    downloaded = Path(
                        hf_hub_download(
                            repo_id=UNSW_HF_REPO,
                            filename=UNSW_HF_FILE,
                            repo_type="dataset",
                            revision=UNSW_HF_REVISION,
                            local_dir=str(raw_dir),
                            force_download=force,
                        )
                    )
                    if downloaded.resolve() != target.resolve():
                        tmp = target.with_suffix(target.suffix + ".copying")
                        tmp.unlink(missing_ok=True)
                        shutil.copy2(downloaded, tmp)
                        tmp.replace(target)

                    ok, why = _looks_like_unsw_csv(target)
                    if ok:
                        break
                    last_error = AssertionError(why)
                    target.unlink(missing_ok=True)
                except Exception as exc:
                    last_error = exc
                    target.unlink(missing_ok=True)
            else:
                raise RuntimeError(
                    "Could not obtain a valid CICFlowMeter_out.csv from the pinned mirror"
                ) from last_error

        # Hash once per valid cached copy. The marker avoids hashing 1.9 GB on
        # every rerun; deleting/replacing the CSV invalidates the marker above.
        marker_ok = (
            marker.is_file()
            and marker.read_text(encoding="utf-8").strip() == UNSW_EXPECTED_SHA256
        )
        if not marker_ok:
            print("  verifying CIC-UNSW-NB15 SHA-256...")
            observed = _sha256_stream(target)
            if observed != UNSW_EXPECTED_SHA256:
                target.unlink(missing_ok=True)
                marker.unlink(missing_ok=True)
                raise AssertionError(
                    "CICFlowMeter_out.csv SHA-256 mismatch: "
                    f"observed {observed}, expected {UNSW_EXPECTED_SHA256}"
                )
            marker.write_text(UNSW_EXPECTED_SHA256 + "\n", encoding="utf-8")

        print(
            f"  raw CIC-UNSW file ready: {target} "
            f"({target.stat().st_size:,} bytes; SHA-256 verified)"
        )

    def _patched_unsw_clean(self):
        csv_path = Path(self.raw_data_dir) / UNSW_HF_FILE
        ok, why = _looks_like_unsw_csv(csv_path)
        if not ok:
            raise AssertionError(
                f"expected a validated {UNSW_HF_FILE} at {csv_path}, but found: {why}"
            )
        df = pd.read_csv(csv_path)
        if "Label" not in df.columns:
            raise KeyError(
                f"Label column missing from {csv_path}; first columns: {list(df.columns[:8])}"
            )
        labels = df["Label"].copy()
        df = df.select_dtypes(include=["number"]).copy()
        df["Label"] = labels.to_numpy()
        return df

    CIC_UNSW._download = _patched_unsw_download
    CIC_UNSW._clean = _patched_unsw_clean

    # Clear only an in-memory failed/partial UNSW entry, if one somehow exists.
    if "_cache" in globals():
        _cache.pop(("unsw", False), None)
        _cache.pop(("unsw", None), None)

    print("CIC-UNSW download patch installed")
else:
    print("CIC-UNSW patch not needed because repository-dependent experiments are disabled")


In [ ]:
CIC_FILES = ["Monday-WorkingHours.pcap_ISCX.csv","Tuesday-WorkingHours.pcap_ISCX.csv",
 "Wednesday-workingHours.pcap_ISCX.csv","Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
 "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv","Friday-WorkingHours-Morning.pcap_ISCX.csv",
 "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv","Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"]
LOCAL_CIC = None      # set to a folder with the 8 raw CSVs to skip the network

def _prefetch_cic(ds):
    """Place the raw CSVs where the loader expects them.

    The repository's own _download validates only the HTTP status code, so an
    error page from the CIC host is written as a .zip and zipfile then fails.
    Creating processed_data_dir also stops _collect from attempting a download.
    """
    raw, proc = Path(ds.raw_data_dir), Path(ds.processed_data_dir)
    raw.mkdir(parents=True, exist_ok=True); proc.mkdir(parents=True, exist_ok=True)
    if Path(ds.file_path).is_file() or len(list(raw.glob("*.csv"))) >= 8: return
    if LOCAL_CIC:
        for p in Path(LOCAL_CIC).rglob("*.csv"): shutil.copy2(p, raw/p.name)
        if len(list(raw.glob("*.csv"))) >= 8: return
    from huggingface_hub import hf_hub_download
    for fn in CIC_FILES:
        shutil.copy2(hf_hub_download(repo_id="c01dsnap/CIC-IDS2017", filename=fn,
                                     repo_type="dataset"), raw/fn)
    print(f"  fetched {len(list(raw.glob('*.csv')))} raw CSVs")

_cache = {}
def load_base(which, grouped):
    """Cleaned table through the repository loader. PCA is never applied here."""
    key = (which, grouped)
    if key in _cache: return _cache[key]
    ds = CICIDS2017(pca=False, classes_mapping=grouped) if which=="cic" else CIC_UNSW(pca=False)
    if which == "cic": _prefetch_cic(ds)
    ds.load()
    df = ds.data.copy().reset_index(drop=True)
    for c in df.columns:
        if df[c].dtype == np.float64: df[c] = df[c].astype(np.float32)
    _cache[key] = df; gc.collect()
    print(f"  {which} grouped={grouped}: {len(df):,} rows, {df.Label.nunique()} classes")
    return df

# ---- protocol, identical to the reported runs ---------------------------
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

TEST_SIZE, FIXED_SEED = 0.30, 2026
KEEP_FRACTION, CAP_MIN = 0.05, 1500
RARE_SHARE = 0.005

def set_seed(s):
    os.environ["PYTHONHASHSEED"]=str(s); np.random.seed(s)
    try:
        import torch, random; random.seed(s); torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception: pass

def split(df, seed):
    """One stratified split, fixed across seeds, as in the reported protocol."""
    vc = df.Label.value_counts(); tiny = list(vc[vc<2].index)
    if tiny: df = df[~df.Label.isin(tiny)]
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, test_size=TEST_SIZE, random_state=FIXED_SEED,
                              stratify=df.Label.to_numpy())
    return df.iloc[tr], df.iloc[te], {"split_seed": FIXED_SEED}

def balanced(df, n, seed):
    r = np.random.default_rng(seed)
    return pd.concat([p.iloc[r.choice(len(p), min(len(p), n), replace=False)]
                      for _, p in df.groupby("Label", sort=True)]).sample(frac=1, random_state=seed)

def cap_test(test, seed):
    """Class-stratified subsample with inverse sampling weights."""
    r = np.random.default_rng(seed); keep=[]; full = test.Label.value_counts()
    for c, p in test.groupby("Label", sort=True):
        k = min(len(p), max(CAP_MIN, int(np.ceil(len(p)*KEEP_FRACTION))))
        keep.append(p.iloc[r.choice(len(p), k, replace=False)] if k < len(p) else p)
    sub = pd.concat(keep); kept = sub.Label.value_counts()
    return sub, {c: float(full[c])/float(kept[c]) for c in kept.index}

def transform(train, test, use_pca=False):
    xtr = train.drop(columns=["Label"]).select_dtypes(include=[np.number]).astype(np.float32)
    xte = test.reindex(columns=xtr.columns).astype(np.float32)
    imp = SimpleImputer(strategy="median"); a=imp.fit_transform(xtr); b=imp.transform(xte)
    vt = VarianceThreshold(0.0); a=vt.fit_transform(a); b=vt.transform(b)
    names = [c for c,k in zip(xtr.columns, vt.get_support()) if k]
    if use_pca:
        sc=StandardScaler(); a=sc.fit_transform(a); b=sc.transform(b)
        k=max(1,min(len(names)//2,len(a)-1))
        pca=IncrementalPCA(n_components=k,batch_size=max(1000,2*k))
        a=pca.fit_transform(a); b=pca.transform(b); names=[f"PC{i+1}" for i in range(k)]
    F=lambda m: pd.DataFrame(np.asarray(m,np.float32),columns=names).reset_index(drop=True)
    return F(a), F(b)

def weighted_metrics(y_true, y_pred, weights=None):
    """Per-class metrics from a confusion matrix rescaled by class weights."""
    labs = sorted(set(map(str,y_true)) | set(map(str,y_pred)))
    idx = {c:i for i,c in enumerate(labs)}
    cm = np.zeros((len(labs),len(labs))); w = weights or {}
    for a,b in zip(map(str,y_true), map(str,y_pred)):
        cm[idx[a], idx[b]] += float(w.get(a,1.0))
    tp=np.diag(cm); col=cm.sum(0); row=cm.sum(1)
    prec=np.divide(tp,col,out=np.zeros_like(tp),where=col>0)
    rec =np.divide(tp,row,out=np.zeros_like(tp),where=row>0)
    f1  =np.divide(2*prec*rec,prec+rec,out=np.zeros_like(tp),where=(prec+rec)>0)
    per = pd.DataFrame({"class":labs,"precision":prec,"recall":rec,"f1":f1,"support":row})
    return float(tp.sum()/cm.sum()), float(f1.mean()), per

def rare_recall(per):
    """Mean recall over classes below RARE_SHARE of the partition, excluding benign."""
    tot = per.support.sum()
    ben = [c for c in per["class"] if str(c).lower()=="benign"]
    r = per[(~per["class"].isin(ben)) & (per.support/tot < RARE_SHARE)]
    return float(r.recall.mean()) if len(r) else np.nan
print("protocol helpers ready")

## Experiment 5 — size-by-balance factorial on CIC-UNSW-NB15 $V_1$

The same four-arm design as the reported CIC-IDS2017 $V_3$ experiment, on the second source.
The small budget equals the matched class-balanced size of the strict panel on this cell
(14,229 rows); the large budget is 400,000. The balanced arms oversample the rarest classes with
replacement, and the realised size, unique-row count and oversampling factor are recorded.
Output: `exp5_size_by_balance_unsw.csv` (same schema as `exp1_size_by_balance.csv`).

In [ ]:
if RUN_2X2_UNSW:
    from xgboost import XGBClassifier
    CELL_WHICH, CELL_GROUPED, USE_PCA = "unsw", False, False   # CIC-UNSW-NB15 V1
    N_SMALL, N_LARGE = 14_229, 400_000   # matched balanced size on this cell

    base = load_base(CELL_WHICH, CELL_GROUPED)

    def natural(df, n, seed):
        return df.sample(min(n, len(df)), random_state=seed)

    def balanced_to(df, n_total, seed):
        """Class-balanced sample of about n_total records.

        A balanced sample drawn without replacement is bounded by the rarest
        class: on CIC-IDS2017 that is Heartbleed with a handful of training
        records, so a large balanced arm is impossible without oversampling and
        the 2x2 would degenerate into a 1x2. Classes with fewer records than the
        per-class quota are therefore oversampled with replacement, which is what
        a practitioner balancing a corpus would do. The realised size and the
        oversampling factor are recorded so the arm is never mistaken for an
        independent sample.
        """
        counts = df.Label.value_counts()
        per = max(1, n_total // len(counts))
        r = np.random.default_rng(seed); parts = []
        for _, p in df.groupby("Label", sort=True):
            idx = r.choice(len(p), per, replace=len(p) < per)
            parts.append(p.iloc[idx])
        return pd.concat(parts)

    RES = OUT/"exp5_size_by_balance_unsw.csv"
    EXPECTED = {"seed","arm","size","balance","n_train","n_unique_rows",
                "oversample_factor","min_class","max_class","accuracy","macro_f1",
                "rare_recall","fit_s"}
    rows, done = [], set()
    if RES.exists():
        prev = pd.read_csv(RES)
        if EXPECTED.issubset(prev.columns):          # resume
            rows = prev.to_dict("records")
            done = set(map(tuple, prev[["seed","arm"]].values))
            print(f"resuming: {len(done)} arms already complete")
        else:                                        # schema changed: start fresh
            RES.rename(RES.with_suffix(".csv.bak"))
            print(f"existing {RES.name} has an older schema "
                  f"(missing {sorted(EXPECTED - set(prev.columns))}); "
                  f"moved to {RES.name}.bak and starting fresh")

    for seed in SEEDS:
        tr, te, _ = split(base, seed)
        te_c, w = cap_test(te, seed)
        yte = te_c.Label.astype(str).reset_index(drop=True)
        arms = {"small_balanced": lambda: balanced_to(tr, N_SMALL, seed),
                "small_natural":  lambda: natural(tr, N_SMALL, seed),
                "large_balanced": lambda: balanced_to(tr, N_LARGE, seed),
                "large_natural":  lambda: natural(tr, N_LARGE, seed)}
        for arm, mk in arms.items():
            if (seed, arm) in done: continue
            sel = mk()
            xtr, xte = transform(sel, te_c, USE_PCA)
            ytr = sel.Label.astype(str).reset_index(drop=True)
            le = LabelEncoder().fit(ytr)
            set_seed(seed)
            m = XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1,
                              subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                              random_state=seed, n_jobs=-1, eval_metric="mlogloss")
            t0=time.time(); m.fit(xtr.to_numpy(), le.transform(ytr)); ft=time.time()-t0
            pred = le.inverse_transform(m.predict(xte.to_numpy()))
            acc, mf1, per = weighted_metrics(yte, pred, w)
            vc = sel.Label.value_counts()
            rows.append({"seed":seed,"arm":arm,
                         "size":"small" if arm.startswith("small") else "large",
                         "balance":"balanced" if arm.endswith("balanced") else "natural",
                         "n_train":len(sel),"n_unique_rows":sel.index.nunique(),
                         "oversample_factor":round(len(sel)/max(1,sel.index.nunique()),2),
                         "min_class":int(vc.min()),"max_class":int(vc.max()),
                         "accuracy":acc,"macro_f1":mf1,
                         "rare_recall":rare_recall(per),"fit_s":ft})
            pd.DataFrame(rows).to_csv(RES, index=False)
            print(f"  seed {seed} {arm:15} n={len(sel):>7,} acc={acc:.4f} "
                  f"rare_recall={rows[-1]['rare_recall']:.4f} ({ft:.0f}s)")

    E = pd.DataFrame(rows)
    print("\ncell means:")
    print(E.groupby(["size","balance"])[["n_train","n_unique_rows","oversample_factor",
                                         "accuracy","rare_recall"]].mean().round(4).to_string())
    if (E[E.balance=="balanced"].oversample_factor > 1.05).any():
        print("\nNote: the balanced arms oversample the rarest classes with replacement,")
        print("so their unique-row count is below their nominal size. Report both.")
    print("\nmain effects on rare-class recall, paired over seeds:")
    for fix in ("size","balance"):
        other = "balance" if fix=="size" else "size"
        for lvl in sorted(E[fix].unique()):
            g = E[E[fix]==lvl].pivot_table(index="seed", columns=other, values="rare_recall")
            if g.shape[1]==2:
                a,b = list(g.columns)
                d = (g[a]-g[b]).dropna().to_numpy()
                if len(d) < 2: continue
                lo,hi = ci(d)
                print(f"  holding {fix}={lvl:9} : {a} - {b} = {d.mean():+.4f} "
                      f"[{lo:+.4f},{hi:+.4f}] p={sf(d):.4f}")
    print("\nRead the 'holding size' rows for the class-balance effect and the")
    print("'holding balance' rows for the record-count effect. If balance is large at")
    print("both sizes and size small at both balances, Section 4.2 should describe a")
    print("class-balance effect and the confounding caveat can be removed.")
else:
    print("Experiment 5 disabled.")

## Experiment 6 — data-rich tree references on the Axis-2 evaluation rows

The matched 45-row anchor answers the controlled question; it cannot show whether a language
model is competitive with a realistically trained detector. This cell rebuilds the exact
Axis-2 split of the released campaign (same Kaggle table, same `three_way` logic, same seeds),
**verifies** that the re-derived evaluation labels equal the persisted `y_true` of every seed and
that the re-fitted 45-row anchor reproduces the persisted anchor accuracy, and then scores two
data-rich references on the identical evaluation rows and the persisted twelve columns:

* **development-rich**: in-context rows ∪ development partition (45 + ≈3,200 rows);
* **large natural**: in-context rows ∪ a natural-distribution sample of up to 400,000 rows
  drawn from every row that is neither in-context nor evaluation.

The in-context rows are included because Heartbleed (11 cleaned records) is exhausted by the
in-context and evaluation partitions: without them no data-rich reference could see that class.
Each reference uses the strong Random Forest and the XGBoost of the Axis-1 panel.
Outputs: `exp6_axis2_datarich.csv`, `exp6_axis2_datarich_perclass.csv`.

In [ ]:
if RUN_AXIS2_DATARICH:
    import kagglehub
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.preprocessing import LabelEncoder
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
    from xgboost import XGBClassifier

    # --- data preparation, identical to axis2_six_model_campaign.ipynb (no subsampling) ----
    CIC_MAP = {"BENIGN":"BENIGN","Benign":"BENIGN","DDoS":"DDoS","DoS Hulk":"DoS",
               "DoS GoldenEye":"DoS","DoS slowloris":"DoS","DoS Slowhttptest":"DoS",
               "PortScan":"Port Scan","Portscan":"Port Scan","FTP-Patator":"Brute Force",
               "SSH-Patator":"Brute Force","Bot":"Bot","Infiltration":"Infiltration",
               "Heartbleed":"Heartbleed"}
    def group(v):
        s = re.sub(r"[^\x00-\x7F]+", " ", str(v)).replace("-", " ")
        s = re.sub(r"\s+", " ", s).strip()
        if "web attack" in s.lower(): return "Web Attack"
        return CIC_MAP.get(s, CIC_MAP.get(str(v).strip(), s))

    root = Path(kagglehub.dataset_download("dhoogla/cicids2017"))
    files = sorted(list(root.rglob("*.parquet")) + list(root.rglob("*.csv")))
    A2 = pd.concat([(pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f, low_memory=True))
                    for f in files], ignore_index=True)
    A2.columns = [str(c).strip().replace(" ", "_") for c in A2.columns]
    lab = "ClassLabel" if "ClassLabel" in A2.columns else "Label"
    A2["Label"] = A2[lab].map(group)
    num = A2.select_dtypes(include=[np.number]).columns.tolist()
    A2 = A2[num + ["Label"]].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    print(f"{len(A2):,} rows, {A2.Label.nunique()} classes")

    K_FEATURES, K_SHOTS, N_TEST_PER_CLASS, N_DEV_PER_CLASS, N_LARGE = 12, 5, 30, 400, 400_000

    def three_way_main(d, seed):
        rng = np.random.default_rng(seed)
        dev, icl, test = [], [], []
        for _, part in d.groupby("Label", sort=True):
            part = part.iloc[rng.permutation(len(part))]
            n_dev = min(N_DEV_PER_CLASS, max(0, len(part) - K_SHOTS - N_TEST_PER_CLASS))
            rest = len(part) - n_dev
            n_icl = min(K_SHOTS, rest); n_test = min(N_TEST_PER_CLASS, max(0, rest - n_icl))
            dev.append(part.iloc[:n_dev]); icl.append(part.iloc[n_dev:n_dev+n_icl])
            test.append(part.iloc[n_dev+n_icl:n_dev+n_icl+n_test])
        cat = lambda ps: pd.concat([p for p in ps if len(p)])   # keep original index for exclusion
        return cat(dev), cat(icl), cat(test)

    def mi_features(d, k, seed):
        x = d.drop(columns=["Label"], errors="ignore").select_dtypes(include=[np.number])
        y = LabelEncoder().fit_transform(d["Label"].astype(str))
        return list(x.columns[np.argsort(mutual_info_classif(x, y, random_state=seed))[::-1][:k]])

    # --- persisted record of the released campaign (for verification and columns) --------
    REPO = Path(REPO_DIR).resolve()
    RAW = REPO/"results"/"axis2"/"raw_generations_original_six_model_campaign.jsonl"
    RESCSV = REPO/"results"/"axis2"/"results_original_six_model_campaign.csv"
    assert RAW.exists() and RESCSV.exists(), "clone the released repository into REPO_DIR first"
    raw = [json.loads(l) for l in RAW.read_text().splitlines() if l.strip()]
    persisted = {}
    for r in raw:
        persisted.setdefault(int(r["seed"]), {"y_true": [str(v) for v in r["y_true"]],
                                              "features": list(r["features"])})
    anchor_ref = (pd.read_csv(RESCSV).query("model == 'RandomForest'")
                    .drop_duplicates("seed").set_index("seed"))

    def make_xgb(seed):
        return XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
                             colsample_bytree=0.9, tree_method="hist", random_state=seed,
                             n_jobs=-1, eval_metric="mlogloss")
    def make_rf(seed):
        return RandomForestClassifier(n_estimators=200, max_features="sqrt",
                                      random_state=seed, n_jobs=-1)

    RES = OUT/"exp6_axis2_datarich.csv"
    rows = pd.read_csv(RES).to_dict("records") if RES.exists() else []
    done = {(r["seed"], r["reference"], r["learner"]) for r in rows}
    PC = OUT/"exp6_axis2_datarich_perclass.csv"
    pc_rows = pd.read_csv(PC).to_dict("records") if PC.exists() else []

    for seed in SEEDS:
        dev, icl, test = three_way_main(A2, seed)
        y_true = test["Label"].astype(str).to_numpy()
        # --- verification against the persisted campaign ---------------------------------
        if seed in persisted:
            same_rows = list(y_true) == persisted[seed]["y_true"]
            feats = persisted[seed]["features"]
            feats_local = mi_features(dev, K_FEATURES, seed)
            print(f"seed {seed}: rows={'MATCH' if same_rows else 'MISMATCH'} "
                  f"features={'match' if feats == feats_local else 'differ (persisted columns used)'}")
            assert same_rows, ("The re-derived evaluation labels differ from the persisted campaign; "
                               "the data preparation is not identical. Do not proceed.")
        else:
            feats = mi_features(dev, K_FEATURES, seed)
        labels = sorted(set(y_true))

        # --- anchor reproduction check ---------------------------------------------------
        rf0 = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
        rf0.fit(icl[feats].to_numpy(), icl["Label"].astype(str).to_numpy())
        acc0 = accuracy_score(y_true, rf0.predict(test[feats].to_numpy()))
        if seed in anchor_ref.index:
            print(f"  anchor accuracy re-fitted {acc0:.4f} vs persisted {anchor_ref.loc[seed,'accuracy']:.4f}")

        # --- data-rich fitting sets -------------------------------------------------------
        excluded = set(icl.index) | set(test.index)
        pool = A2.drop(index=list(excluded))
        rng = np.random.default_rng(seed)
        large = pool.iloc[rng.choice(len(pool), min(N_LARGE, len(pool)), replace=False)]
        fits = {"development-rich": pd.concat([icl, dev]),
                "large natural":    pd.concat([icl, large])}
        for ref, F in fits.items():
            for learner, mk in [("strong RF", make_rf), ("XGBoost", make_xgb)]:
                if (seed, ref, learner) in done: continue
                ytr = F["Label"].astype(str).to_numpy()
                le = LabelEncoder().fit(ytr)
                m = mk(seed); t0 = time.time()
                if learner == "XGBoost":
                    m.fit(F[feats].to_numpy(), le.transform(ytr))
                    pred = le.inverse_transform(m.predict(test[feats].to_numpy()))
                else:
                    m.fit(F[feats].to_numpy(), ytr)
                    pred = m.predict(test[feats].to_numpy())
                ft = time.time()-t0
                p_, r_, f_, s_ = precision_recall_fscore_support(y_true, pred, labels=labels,
                                                                 zero_division=0)
                rows.append({"seed": seed, "reference": ref, "learner": learner, "n_fit": len(F),
                             "n_fit_unique": F.index.nunique(), "n_eval": len(test),
                             "accuracy": accuracy_score(y_true, pred),
                             "macro_f1": f1_score(y_true, pred, labels=labels, average="macro",
                                                  zero_division=0),
                             "anchor_accuracy_refit": acc0, "fit_s": ft})
                for lb, pv, rv, fv, sv in zip(labels, p_, r_, f_, s_):
                    pc_rows.append({"seed": seed, "reference": ref, "learner": learner, "class": lb,
                                    "precision": float(pv), "recall": float(rv), "f1": float(fv),
                                    "support": int(sv)})
                pd.DataFrame(rows).to_csv(RES, index=False)
                pd.DataFrame(pc_rows).to_csv(PC, index=False)
                print(f"  {ref:18} {learner:9} n_fit={len(F):>7,} acc={rows[-1]['accuracy']:.4f} "
                      f"macroF1={rows[-1]['macro_f1']:.4f} ({ft:.0f}s)")
    E6 = pd.DataFrame(rows)
    print("\nmeans over seeds:")
    print(E6.groupby(["reference","learner"])[["n_fit","accuracy","macro_f1"]]
            .mean().round(4).to_string())
else:
    print("Experiment 6 disabled.")

## Experiment 4b - ten-seed ManyClass sensitivity

The historical wrapper used by the completed experiment is retained under `src/models/`. The compatibility cell is needed only for an optional expensive rerun with `tabpfn==6.0.6`.


In [ ]:
if RUN_MANYCLASS_10:
    import importlib
    import os

    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN before the ManyClass rerun"
    from scripts.tabpfn_compat import prepare_tabpfn_environment

    print(prepare_tabpfn_environment(require_gpu=True))

    def _import_with_stubs(target):
        return importlib.import_module(target), []
else:
    print("ManyClass model re-execution disabled; the persisted 40-row CSV remains available")


In [ ]:
# ============================================================
#  EXPERIMENT 4 — ManyClass sensitivity (ten seeds)
# ============================================================
if RUN_MANYCLASS_10:
    import importlib

    _models, _stubbed_m = _import_with_stubs("src.models")
    mcfg        = importlib.import_module("src.models.config")
    TabPFNModel = importlib.import_module("src.models.trainable.tabpfn").TabPFNModel
    print(f"src.models OK" + (f"; placeholders: {_stubbed_m}" if _stubbed_m else ""))

    base  = load_base("cic", False)          # CIC-IDS2017 V1, 15 classes
    saved = dict(mcfg.TABPFN_MANY_CLASS_CONFIG)

    RES  = OUT/"exp4_manyclass.csv"
    _rel = Path(REPO_DIR)/"results"/"sensitivity"/"exp4_manyclass.csv"
    if not RES.exists() and _rel.exists():
        shutil.copy2(_rel, RES); print("copied released three-seed record")
    rows = pd.read_csv(RES).to_dict("records") if RES.is_file() else []
    done = {(r["alphabet_size"], r["n_estimators_redundancy"], r["seed"]) for r in rows}
    if done: print(f"resuming: {len(done)} of {4*len(SEEDS)} runs already complete")

    try:
        for alphabet in (5, 10):
            for redundancy in (1, 4):
                for seed in SEEDS:
                    if (alphabet, redundancy, seed) in done: continue
                    # Set the wrapper configuration BEFORE constructing the model:
                    # TabPFNModel reads it at construction, not at fit.
                    mcfg.TABPFN_MANY_CLASS_CONFIG.update(
                        alphabet_size=alphabet, n_estimators_redundancy=redundancy,
                        random_state=seed)
                    mcfg.TABPFN_CONFIG["random_state"] = seed

                    tr, te, _ = split(base, seed)
                    te_c, w   = cap_test(te, seed)
                    sel  = balanced(tr, 2000, seed)
                    xtr, xte = transform(sel, te_c, False)
                    ytr = sel.Label.astype(str).reset_index(drop=True)
                    yte = te_c.Label.astype(str).reset_index(drop=True)

                    set_seed(seed)
                    m = TabPFNModel(name=f"tabpfn_a{alphabet}_r{redundancy}_s{seed}",
                                    version="2.5", use_default_config=False)
                    t0 = time.time(); m.fit(xtr.to_numpy(), ytr.to_numpy()); ft = time.time()-t0
                    pred = np.asarray(m.predict(xte.to_numpy()))
                    acc, mf1, per = weighted_metrics(yte, pred, w)

                    n_classes = int(pd.Series(ytr).nunique())
                    rows.append({"alphabet_size":alphabet,"n_estimators_redundancy":redundancy,
                                 "seed":seed,"n_classes":n_classes,
                                 "wrapper_engaged":n_classes > 10,
                                 "accuracy":acc,"macro_f1":mf1,
                                 "rare_recall":rare_recall(per),"fit_s":ft})
                    pd.DataFrame(rows).to_csv(RES, index=False)
                    print(f"  a={alphabet} r={redundancy} seed={seed} acc={acc:.4f} "
                          f"rare_recall={rows[-1]['rare_recall']:.4f} ({ft:.0f}s)")
                    del m; gc.collect()
    finally:
        mcfg.TABPFN_MANY_CLASS_CONFIG.clear()
        mcfg.TABPFN_MANY_CLASS_CONFIG.update(saved)

    MC = pd.DataFrame(rows)
    # If the wrapper never engaged, the 2x2 measures nothing and must not be read
    # as evidence that the setting is immaterial.
    if not MC.wrapper_engaged.all():
        print(f"\nWARNING: the wrapper engaged on {int(MC.wrapper_engaged.sum())} of "
              f"{len(MC)} runs. It activates above ten classes; this cell fits "
              f"{int(MC.n_classes.iloc[0])}. Results below are not a sensitivity test.")
    print("\n" + MC.groupby(["alphabet_size","n_estimators_redundancy"])
             [["accuracy","macro_f1","rare_recall"]].mean().round(4).to_string())
    g = MC.groupby(["alphabet_size","n_estimators_redundancy"]).rare_recall.mean()
    print(f"\nrare-class recall spread across settings: {g.max()-g.min():.4f}")
    print("Compare against the smallest margin reported in Table 6. If the spread is")
    print("below it, the wrapper is not load-bearing and Section 6.4 can say so with")
    print("evidence rather than by disclosure alone.")
else:
    print("Experiment 4b disabled.")


## Analysis - fitted-model extensions

This section summarises the three local fitted-model experiments. The completed hosted-model panel is reconstructed independently from raw generations by `scripts/recompute_axis2_second_variant.py`.


In [ ]:
GEN = OUT/"generated"; GEN.mkdir(exist_ok=True)
SUMMARY = {}
def write(name, text):
    (GEN/name).write_text(text); print(f"[table] {name}")
    if MANUSCRIPT_DIR:
        d = Path(MANUSCRIPT_DIR)/"generated"; d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(GEN/name, d/name)

# ----------------------------------------------------------- Experiment 5: UNSW factorial
p5 = OUT/"exp5_size_by_balance_unsw.csv"
if p5.exists():
    E = pd.read_csv(p5)
    arms = E.groupby(["size","balance"]).agg(n_train=("n_train","mean"),
                                             n_unique=("n_unique_rows","mean"),
                                             macro_f1=("macro_f1","mean"),
                                             rare_recall=("rare_recall","mean"))
    pv = E.pivot(index="seed", columns="arm", values="rare_recall")
    contrasts = [("Balance, small budget","small_balanced","small_natural"),
                 ("Balance, large budget","large_balanced","large_natural"),
                 ("Size, natural distribution","large_natural","small_natural"),
                 ("Size, balanced distribution","large_balanced","small_balanced")]
    rows = []
    for lab, a, b in contrasts:
        d = (pv[a]-pv[b]).dropna().to_numpy(); lo, hi = ci(d, 50000)
        rows.append({"contrast": lab, "delta": d.mean(), "lo": lo, "hi": hi,
                     "wins": int((d>0).sum()), "n": len(d), "p": sf(d)})
    ph = holm([r["p"] for r in rows])
    for r, q in zip(rows, ph): r["p_holm"] = float(q)
    C = pd.DataFrame(rows); C.to_csv(OUT/"exp5_contrasts.csv", index=False)
    t = ["\\begin{minipage}{\\linewidth}","\\centering","\\textbf{Panel A: arm means.}\\\\[3pt]",
         "\\begin{tabular}{@{}llrrrr@{}}","\\toprule",
         "Budget & Distribution & Rows & Unique rows & Macro-$F_1$ & Rare-class recall \\\\","\\midrule"]
    for (sz, bl) in [("small","natural"),("small","balanced"),("large","natural"),("large","balanced")]:
        r = arms.loc[(sz, bl)]
        t.append(f"{sz.capitalize()} & {bl.capitalize()} & {int(round(r.n_train)):,} & "
                 f"{int(round(r.n_unique)):,} & {r.macro_f1:.3f} & {r.rare_recall:.3f} \\\\")
    t += ["\\bottomrule","\\end{tabular}","\\par\\medskip",
          "\\textbf{Panel B: planned paired contrasts.}\\\\[3pt]",
          "\\begin{tabular}{@{}lrrrr@{}}","\\toprule",
          "Contrast & $\\Delta$ rare-class recall & 95\\% CI & Wins & $p_{\\mathrm{Holm}}$ \\\\","\\midrule"]
    for _, r in C.iterrows():
        t.append(f"{r.contrast} & {r.delta:+.3f} & [{r.lo:+.3f}, {r.hi:+.3f}] & "
                 f"{r.wins}/{r.n} & {r.p_holm:.4f} \\\\")
    t += ["\\bottomrule","\\end{tabular}","\\end{minipage}"]
    write("table_size_balance_unsw.tex", "\n".join(t)+"\n")
    SUMMARY["exp5_unsw_factorial"] = {"arms": arms.reset_index().to_dict("records"),
                                      "contrasts": C.to_dict("records")}
else:
    print("[skip] experiment 5 outputs not found")

# ----------------------------------------------------------- Experiment 4b: ManyClass
p4 = OUT/"exp4_manyclass.csv"
if not p4.exists():
    alt = Path(REPO_DIR)/"results"/"sensitivity"/"exp4_manyclass.csv"
    p4 = alt if alt.exists() else None
if p4 is not None:
    MC = pd.read_csv(p4); rows = []
    for (a, r_), g in MC.groupby(["alphabet_size","n_estimators_redundancy"]):
        x = g.rare_recall.to_numpy()
        lo, hi = tci(x) if len(x) < 10 else ci(x, 50000)
        rows.append({"alphabet": a, "redundancy": r_, "n": len(x),
                     "accuracy": g.accuracy.mean(), "macro_f1": g.macro_f1.mean(),
                     "rare_recall": x.mean(), "lo": lo, "hi": hi})
    M4 = pd.DataFrame(rows)
    kind = "Student-$t$" if M4.n.min() < 10 else "percentile bootstrap"
    t = ["\\begin{tabular}{@{}rrrrrrr@{}}","\\toprule",
         "Alphabet & Redundancy & $n$ & Accuracy & Macro-$F_1$ & Rare-class recall & 95\\% CI \\\\",
         "\\midrule"]
    for _, r in M4.iterrows():
        t.append(f"{int(r.alphabet)} & {int(r.redundancy)} & {int(r.n)} & {r.accuracy:.4f} & "
                 f"{r.macro_f1:.4f} & {r.rare_recall:.4f} & [{r.lo:.4f}, {r.hi:.4f}] \\\\")
    t += ["\\bottomrule","\\end{tabular}"]
    write("table_manyclass.tex", "\n".join(t)+"\n")
    mp = MC.pivot(index="seed", columns=["alphabet_size","n_estimators_redundancy"],
                  values="rare_recall")
    paired = []
    for col in mp.columns:
        if col == (5, 1): continue
        d = (mp[col]-mp[(5,1)]).dropna().to_numpy()
        paired.append({"setting": f"a{col[0]}_r{col[1]}", "delta_vs_a5r1": d.mean(),
                       "p": sf(d), "n": len(d)})
    SUMMARY["manyclass"] = {"n_seeds": int(MC.seed.nunique()), "interval": kind,
                            "spread": float(M4.rare_recall.max()-M4.rare_recall.min()),
                            "settings": M4.to_dict("records"), "paired_vs_a5r1": paired}
    print(f"ManyClass: n_seeds={MC.seed.nunique()} "
          f"spread={M4.rare_recall.max()-M4.rare_recall.min():.4f} ({kind} intervals)")

# ----------------------------------------------------------- Experiment 6: data-rich references
p6 = OUT/"exp6_axis2_datarich.csv"
if p6.exists():
    E6 = pd.read_csv(p6)
    rf_ref = (pd.read_csv(Path(REPO_DIR)/"results"/"axis2"/"results_original_six_model_campaign.csv")
                .query("model=='RandomForest'").drop_duplicates("seed").set_index("seed"))
    repo_res = Path(REPO_DIR)/"results"/"derived"/"axis2_closed_label_per_seed.csv"
    best = None
    if repo_res.exists():
        per = pd.read_csv(repo_res)
        best_name = per.groupby("model").macro_f1_closed.mean().idxmax()
        best = per[per.model == best_name].set_index("seed")
    t = ["\\begin{tabular}{@{}lrrrrrr@{}}","\\toprule",
         "Reference & Fitting rows & Accuracy & Macro-$F_1$ & $\\Delta F_1$ vs.\\ anchor & Wins & $p$ \\\\",
         "\\midrule",
         f"Random Forest anchor (45 rows) & 45 & {rf_ref.accuracy.mean():.3f} & "
         f"{rf_ref.macro_f1.mean():.3f} & -- & -- & -- \\\\"]
    out = []
    for ref in ["development-rich","large natural"]:
        for learner in ["strong RF","XGBoost"]:
            g = E6[(E6.reference==ref)&(E6.learner==learner)].set_index("seed").sort_index()
            if not len(g): continue
            common = sorted(set(g.index)&set(rf_ref.index))
            d = g.loc[common,"macro_f1"].to_numpy()-rf_ref.loc[common,"macro_f1"].to_numpy()
            rec = {"reference": ref, "learner": learner, "n_fit": g.n_fit.mean(),
                   "accuracy": g.accuracy.mean(), "macro_f1": g.macro_f1.mean(),
                   "delta_vs_anchor": d.mean(), "wins": int((d>0).sum()), "n": len(d), "p": sf(d)}
            if best is not None:
                cb = sorted(set(g.index)&set(best.index))
                db = g.loc[cb,"macro_f1"].to_numpy()-best.loc[cb,"macro_f1_closed"].to_numpy()
                rec.update({"best_llm": best_name, "delta_vs_best_llm": db.mean(),
                            "wins_vs_best_llm": int((db>0).sum())})
            out.append(rec)
            lab = {"development-rich":"development partition",
                   "large natural":"large natural sample"}[ref]
            t.append(f"{learner.replace('strong RF','Strong RF')}, {lab} & "
                     f"{int(round(g.n_fit.mean())):,} & {g.accuracy.mean():.3f} & "
                     f"{g.macro_f1.mean():.3f} & {d.mean():+.3f} & {int((d>0).sum())}/{len(d)} & "
                     f"{sf(d):.4f} \\\\")
    t += ["\\bottomrule","\\end{tabular}"]
    write("table_axis2_datarich.tex", "\n".join(t)+"\n")
    SUMMARY["exp6_datarich"] = out
    print(pd.DataFrame(out).round(4).to_string())
else:
    print("[skip] experiment 6 outputs not found")

# The completed hosted-model panel is analysed by scripts/recompute_axis2_second_variant.py.

(OUT/"revision4_summary.json").write_text(json.dumps(SUMMARY, indent=2, default=float))
print("\n=== numbers for the \\pending markers (revision4_summary.json) ===")
print(json.dumps(SUMMARY, indent=2, default=float)[:6000])


In [ ]:
# ============================================================
#  DOWNLOAD — archive everything this notebook produced.
#  Paste as the final cell. Safe to run at any point, including
#  part-way through a long experiment.
# ============================================================
import hashlib, json, os, zipfile
from datetime import datetime, timezone
from pathlib import Path

# Resolve the output directory even if this cell is run on its own.
try:
    OUT
except NameError:
    OUT = Path(os.environ.get("OUT_DIR", "revision4_outputs"))
OUT = Path(OUT)

INCLUDE_SOURCE = True    # bundle the notebook so the archive reproduces itself
INCLUDE_RAW    = True    # raw JSONL files are retained when present
MAX_MB         = 512

if not OUT.exists() or not any(OUT.rglob("*")):
    print(f"[archive] nothing to package: {OUT.resolve()} is empty.")
    print("          Enable an experiment and run it first.")
else:
    files = sorted(p for p in OUT.rglob("*")
                   if p.is_file() and p.suffix not in (".zip",) and
                      (INCLUDE_RAW or p.suffix != ".jsonl"))
    # The cached Axis-2 table is re-derivable from Kaggle; exclude it from the archive.
    files = [p for p in files if p.name != "axis2_table.parquet"]

    extras = []
    if INCLUDE_SOURCE:
        for name in ("revision4_experiments.ipynb", "scripts/recompute_manuscript_results.py"):
            for base in (Path.cwd(), Path("/content")):
                c = (base / name)
                if c.is_file() and c.resolve() not in extras:
                    extras.append(c.resolve()); break

    total_mb = sum(p.stat().st_size for p in files + extras) / 1e6
    print(f"[archive] {len(files)} result files + {len(extras)} source files, {total_mb:.2f} MB")

    if total_mb > MAX_MB:
        print(f"[archive] ABORTED: exceeds MAX_MB={MAX_MB}. Largest files:")
        for p in sorted(files, key=lambda q: -q.stat().st_size)[:8]:
            print(f"            {p.stat().st_size/1e6:8.1f} MB  {p.relative_to(OUT)}")
        print("          Raise MAX_MB, or set INCLUDE_RAW = False to drop the generations.")
    else:
        # SHA-256 manifest, matching the discipline of the experiment record, so a
        # recipient can verify that what they received is what was produced.
        manifest = {"generated_utc": datetime.now(timezone.utc).isoformat(),
                    "output_dir": str(OUT.resolve()),
                    "experiments": {}, "n_files": len(files), "files": []}
        for flag, name in (("RUN_2X2_UNSW","exp5_size_by_balance_unsw.csv"),
                           ("RUN_AXIS2_DATARICH","exp6_axis2_datarich.csv"),
                           ("RUN_MANYCLASS_10","exp4_manyclass.csv")):
            manifest["experiments"][flag] = {
                "enabled": bool(globals().get(flag, False)),
                "output_present": (OUT/name).is_file()}
        for p in files + extras:
            h = hashlib.sha256()
            with p.open("rb") as fh:
                for b in iter(lambda: fh.read(1 << 20), b""):
                    h.update(b)
            manifest["files"].append({
                "path": str(p.relative_to(OUT)) if p.is_relative_to(OUT) else f"source/{p.name}",
                "bytes": p.stat().st_size, "sha256": h.hexdigest()})
        mpath = OUT / "ARCHIVE_MANIFEST.json"
        mpath.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

        stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
        archive = Path(f"sensitivity_experiments_{stamp}.zip")
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
            for p in files + [mpath]:
                z.write(p, arcname=str(Path("revision4_outputs") / p.relative_to(OUT)))
            for p in extras:
                z.write(p, arcname=f"source/{p.name}")

        by_ext = {}
        for p in files:
            by_ext[p.suffix or "(none)"] = by_ext.get(p.suffix or "(none)", 0) + 1
        print(f"[archive] wrote {archive}  ({archive.stat().st_size/1e6:.2f} MB, "
              f"{len(files)+len(extras)+1} entries)")
        print("[archive] contents: " + ", ".join(f"{k} x{v}" for k, v in
                                                 sorted(by_ext.items(), key=lambda kv: -kv[1])))
        for flag, info in manifest["experiments"].items():
            if info["enabled"] and not info["output_present"]:
                print(f"[archive] WARNING: {flag} was enabled but produced no output file")

        delivered = False
        try:
            from google.colab import files as _cf
            _cf.download(str(archive)); delivered = True
            print("[archive] browser download started.")
        except Exception:
            pass
        if not delivered:
            try:
                from IPython.display import FileLink, display
                display(FileLink(str(archive)))
            except Exception:
                pass
        print(f"[archive] absolute path: {archive.resolve()}")
